# Preprocesamiento - Capa SILVER

**Objetivo:** parsear y validar los archivos crudos de `bronze/`, estandarizar
formatos, eliminar duplicados/errores y **unir** todos los meses en un dataset
consistente y reutilizable.

## Lo que encontramos en bronze

- 3 generaciones de esquema con nombres de columna distintos (legacy `;`,
  legacy `,` y moderno `Codigo/CodigoUbigeo/...`) mas un xlsx.
- Encoding **mixto**: cabeceras cp1252, cuerpo con `Ñ` en cp850 (`0xA5`) y
  algunos archivos UTF-8 (incluso doblemente codificados). Hay BOM en los zip.
- Un archivo **sin cabecera** (2023-10) y uno en **xlsx** con una fila inicial
  vacia (2024-12).
- **Dos granularidades**: 2023-01..2025-12 = una fila por suministro;
  2026-01..2026-08 = filas **agregadas** con el conteo `Clientes`.
- `codigo_cliente` es un **correlativo denso (1..N) por archivo**: NO es un
  identificador estable del suministro entre meses.
- El `ubigeo` **no siempre coincide** con departamento/provincia/distrito.
- Hay `fecha_alta` con **fechas centinela** (ej. 1908-09-27).

## Salidas

| archivo | contenido |
|---|---|
| `silver/consumo_clientes.parquet` | serie 2023-01..2025-12, 1 fila por (correlativo, periodo) |
| `silver/consumo_agregado.parquet` | 2026-01..2026-08, filas agregadas |
| `silver/resumen_mensual.parquet` | totales por periodo y granularidad |
| `silver/_rechazados.parquet` | filas descartadas por validacion |
| `silver/_reporte_silver.json` | reporte de validacion |

## 0. Imports y rutas

In [3]:
import io
import json
import re
import shutil
import unicodedata
import zipfile
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 200)

HERE = Path.cwd()
PREP = HERE if HERE.name == "preprocesamiento" else HERE / "preprocesamiento"
BRONZE = PREP / "bronze"
SILVER = PREP / "silver"
PARTS = SILVER / "_parts"

if PARTS.exists():
    shutil.rmtree(PARTS)
PARTS.mkdir(parents=True, exist_ok=True)
SILVER.mkdir(parents=True, exist_ok=True)

print("bronze:", BRONZE.resolve())
print("silver:", SILVER.resolve())

bronze: D:\PROYECTO MAESTRIA\CURSO II - SEMESTRE II\visualizacion-datos-trabajo\preprocesamiento\bronze
silver: D:\PROYECTO MAESTRIA\CURSO II - SEMESTRE II\visualizacion-datos-trabajo\preprocesamiento\silver


## 1. Esquema canonico y mapa de alias

Todas las variantes de nombre se mapean a un unico conjunto de columnas.

In [4]:
CANON = [
    "codigo_cliente", "ubigeo", "departamento", "provincia", "distrito",
    "fecha_alta", "tarifa", "periodo", "consumo_kwh", "facturacion_soles",
    "estado_cliente", "fecha_corte", "clientes",
]

REQUIRED = ["ubigeo", "departamento", "provincia", "distrito",
            "tarifa", "periodo", "consumo_kwh", "facturacion_soles",
            "estado_cliente"]

# nombre normalizado (sin acentos, minuscula, solo alfanumerico) -> columna canonica
ALIASES = {
    "codigo_cliente": {"codigocliente", "correlativo", "codigo"},
    "ubigeo": {"ubigeo", "codigoubigeo"},
    "departamento": {"departamento", "nombredepartamento"},
    "provincia": {"provincia", "nombreprovincia"},
    "distrito": {"distrito", "nombredistrito"},
    "fecha_alta": {"fechaalta", "iniciocontrato"},
    "tarifa": {"tarifa"},
    "periodo": {"periodo", "codigoperiodocomercial"},
    "consumo_kwh": {"consumo", "cea"},
    "facturacion_soles": {"facturacion", "mtotal"},
    "estado_cliente": {"estadocliente", "nombreestadosuministro"},
    "fecha_corte": {"fechacorte", "fecha"},
    "clientes": {"clientes"},
}
KNOWN_TOKENS = set().union(*ALIASES.values())

# orden posicional del esquema legacy (para el archivo sin cabecera)
LEGACY_ORDER = ["codigo_cliente", "ubigeo", "departamento", "provincia",
                "distrito", "fecha_alta", "tarifa", "periodo", "consumo_kwh",
                "facturacion_soles", "estado_cliente", "fecha_corte"]


def norm_name(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", s.lower())


def detect_sep(line):
    return max([";", ",", "\t"], key=lambda d: line.count(d))


def looks_like_header(line, sep):
    fields = [norm_name(x) for x in line.split(sep)]
    return sum(1 for f in fields if f in KNOWN_TOKENS) >= 3

## 2. Lectura normalizada (encoding mixto)

Pipeline de texto: intenta UTF-8 (con BOM); si falla, latin-1. Luego repara el
`Ñ` que quedo como `0xA5`/`¥` (cp850).

In [5]:
def normalize_text(raw: bytes) -> str:
    try:
        s = raw.decode("utf-8-sig")
    except UnicodeDecodeError:
        s = raw.decode("latin-1")
    return s.replace("\xa5", "\u00d1")  # '¥' -> 'Ñ'


def read_csv_bytes(raw: bytes) -> pd.DataFrame:
    text = normalize_text(raw)
    line = text.splitlines()[0]
    sep = detect_sep(line)
    if looks_like_header(line, sep):
        df = pd.read_csv(io.StringIO(text), sep=sep, dtype=str,
                         keep_default_na=False, na_values=[""],
                         engine="python")
    else:
        df = pd.read_csv(io.StringIO(text), sep=sep, header=None, dtype=str,
                         keep_default_na=False, na_values=[""],
                         engine="python", names=LEGACY_ORDER)
    df = df.loc[:, [c for c in df.columns
                    if not str(c).lower().startswith("unnamed")]]
    return df


def read_xlsx_bytes(raw: bytes) -> pd.DataFrame:
    import openpyxl  # noqa: F401

    tmp = SILVER / "_tmp.xlsx"
    tmp.write_bytes(raw)
    try:
        # este archivo trae una fila inicial vacia => cabecera en la fila 1
        df = pd.read_excel(tmp, header=1, dtype=str, engine="openpyxl")
    finally:
        tmp.unlink(missing_ok=True)
    df = df.loc[:, [c for c in df.columns
                    if not str(c).lower().startswith("unnamed")]]
    return df


def read_any(path: Path) -> pd.DataFrame:
    if path.suffix == ".csv":
        return read_csv_bytes(path.read_bytes())
    with zipfile.ZipFile(path) as z:
        member = z.namelist()[0]
        raw = z.read(member)
    if member.lower().endswith(".csv"):
        return read_csv_bytes(raw)
    if member.lower().endswith(".xlsx"):
        return read_xlsx_bytes(raw)
    raise ValueError(f"formato no soportado: {member}")

## 3. Estandarizacion a esquema canonico

In [6]:
def parse_fecha(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    out = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
    mask = out.isna() & s.notna()
    if mask.any():
        out = out.copy()
        out.loc[mask] = pd.to_datetime(s[mask], errors="coerce", format="mixed")
    return out


def to_canonical(df: pd.DataFrame, source: str) -> pd.DataFrame:
    ren = {}
    for col in df.columns:
        n = norm_name(col)
        for canon, alts in ALIASES.items():
            if n in alts:
                ren[col] = canon
                break
    df = df.rename(columns=ren)
    granularidad = "agregado" if "clientes" in df.columns else "cliente"

    for c in CANON:
        if c not in df.columns:
            df[c] = pd.NA
    df = df[CANON].copy()

    df["codigo_cliente"] = pd.to_numeric(df["codigo_cliente"], errors="coerce").astype("Int64")
    df["ubigeo"] = (df["ubigeo"].astype("string").str.strip()
                     .str.replace(r"\.0$", "", regex=True).str.zfill(6))
    for c in ["departamento", "provincia", "distrito", "tarifa", "estado_cliente"]:
        df[c] = (df[c].astype("string").str.strip()
                 .str.replace("\xa5", "\u00d1", regex=False)  # '¥' -> 'Ñ' (xlsx)
                 .str.upper())
    df["periodo"] = pd.to_numeric(df["periodo"], errors="coerce").astype("Int32")
    df["consumo_kwh"] = pd.to_numeric(df["consumo_kwh"], errors="coerce").astype("float64")
    df["facturacion_soles"] = pd.to_numeric(df["facturacion_soles"], errors="coerce").astype("float64")
    df["clientes"] = pd.to_numeric(df["clientes"], errors="coerce").astype("Int32")
    df["fecha_alta"] = parse_fecha(df["fecha_alta"])
    df["fecha_corte"] = parse_fecha(df["fecha_corte"])
    df["granularidad"] = granularidad
    df["source_file"] = Path(source).name
    return df

## 4. Procesar cada archivo -> un parquet por periodo

Se procesa **de a un archivo** y se vuelca a `silver/_parts/` con DuckDB para
mantener el uso de memoria bajo.

In [7]:
PROJECTION = """
SELECT
    codigo_cliente::BIGINT     AS codigo_cliente,
    ubigeo,
    departamento, provincia, distrito,
    tarifa, estado_cliente,
    periodo::INTEGER           AS periodo,
    CAST(fecha_alta  AS DATE)  AS fecha_alta,
    consumo_kwh::DOUBLE        AS consumo_kwh,
    facturacion_soles::DOUBLE  AS facturacion_soles,
    CAST(fecha_corte AS DATE)  AS fecha_corte,
    clientes::INTEGER          AS clientes,
    granularidad, source_file
FROM _tmp
"""

con = duckdb.connect()
log = []

data_files = [p for p in sorted(BRONZE.iterdir())
              if not p.name.startswith("_")
              and p.suffix in (".csv", ".zip")]

for path in data_files:
    period_label = path.stem.replace("ELPU_", "")
    try:
        raw_df = read_any(path)
        df = to_canonical(raw_df, path.name)
        del raw_df

        con.register("_tmp", df)
        part = PARTS / f"{period_label}.parquet"
        con.execute(f"COPY ({PROJECTION}) TO '{part}' "
                    "(FORMAT PARQUET, COMPRESSION ZSTD)")
        con.unregister("_tmp")

        periodos = sorted(int(x) for x in df["periodo"].dropna().unique())
        log.append({
            "archivo": path.name,
            "periodo_archivo": period_label,
            "filas": len(df),
            "periodos_datos": periodos,
            "granularidad": df["granularidad"].iloc[0],
            "columnas_ok": True,
        })
        print(f"OK  {path.name:16} filas={len(df):>7,}  "
              f"periodo={periodos}  {df['granularidad'].iloc[0]}")
        del df
    except Exception as exc:  # noqa: BLE001
        log.append({"archivo": path.name, "error": str(exc), "columnas_ok": False})
        print(f"ERR {path.name:16} {exc}")

procesados = pd.DataFrame(log)
print(f"\nArchivos procesados: {len(procesados)}")

OK  ELPU_2023-01.csv filas=336,736  periodo=[202301]  cliente
OK  ELPU_2023-02.csv filas=336,784  periodo=[202302]  cliente
OK  ELPU_2023-03.csv filas=336,988  periodo=[202303]  cliente
OK  ELPU_2023-04.csv filas=337,450  periodo=[202304]  cliente
OK  ELPU_2023-05.csv filas=338,474  periodo=[202305]  cliente
OK  ELPU_2023-06.zip filas=339,871  periodo=[202306]  cliente
OK  ELPU_2023-07.zip filas=340,633  periodo=[202307]  cliente
OK  ELPU_2023-08.csv filas=341,198  periodo=[202308]  cliente
OK  ELPU_2023-09.csv filas=341,642  periodo=[202309]  cliente
OK  ELPU_2023-10.csv filas=342,269  periodo=[202310]  cliente
OK  ELPU_2023-11.zip filas=342,839  periodo=[202311]  cliente
OK  ELPU_2023-12.zip filas=343,082  periodo=[202312]  cliente
OK  ELPU_2024-01.csv filas=343,447  periodo=[202401]  cliente
OK  ELPU_2024-02.zip filas=344,103  periodo=[202402]  cliente
OK  ELPU_2024-03.csv filas=344,496  periodo=[202403]  cliente
OK  ELPU_2024-04.csv filas=345,299  periodo=[202404]  cliente
OK  ELPU

## 5. Union de todos los periodos

In [8]:
GLOB = str(PARTS / "*.parquet")
READ = f"read_parquet('{GLOB}', union_by_name=true)"

total, = con.execute(f"SELECT count(*) FROM {READ}").fetchone()
print(f"Filas totales unidas: {total:,}")
con.execute(f"SELECT granularidad, count(*) AS filas, "
            f"count(DISTINCT periodo) AS n_periodos "
            f"FROM {READ} GROUP BY 1").df()

Filas totales unidas: 12,554,866


,granularidad,filas,n_periodos
0,cliente,12480381,34
1,agregado,74485,8


## 6. Validacion y control de errores

Chequeos: nulos en campos obligatorios, consumos/facturacion negativos, ubigeo
mal formado, tarifas fuera de catalogo, periodo no coincide con el archivo y
duplicados de (cliente, periodo).

In [9]:
checks = {}

checks["nulos_obligatorios"] = con.execute(f"""
    SELECT count(*) AS filas
    FROM {READ}
    WHERE ({' OR '.join(f'{c} IS NULL' for c in REQUIRED)})
""").df()

checks["negativos"] = con.execute(f"""
    SELECT count(*) AS filas
    FROM {READ}
    WHERE consumo_kwh < 0 OR facturacion_soles < 0
""").df()

checks["ubigeo_invalido"] = con.execute(f"""
    SELECT count(*) AS filas
    FROM {READ}
    WHERE ubigeo IS NULL OR NOT regexp_matches(ubigeo, '^[0-9]{{6}}$')
""").df()

checks["duplicados_cliente_periodo"] = con.execute(f"""
    SELECT count(*) AS grupos
    FROM (
        SELECT codigo_cliente, periodo
        FROM {READ}
        WHERE granularidad = 'cliente'
        GROUP BY 1, 2 HAVING count(*) > 1
    )
""").df()

checks["tarifas"] = con.execute(f"""
    SELECT tarifa, count(*) AS filas
    FROM {READ} GROUP BY 1 ORDER BY 2 DESC
""").df()

checks["periodo_vs_archivo"] = con.execute(f"""
    SELECT source_file, string_agg(DISTINCT periodo::VARCHAR, ',') AS periodos
    FROM {READ}
    WHERE granularidad = 'cliente'
    GROUP BY 1
""").df()

# archivos que traen el mismo periodo (el portal publico copias duplicadas)
checks["fuentes_duplicadas"] = con.execute(f"""
    SELECT periodo, count(DISTINCT source_file) AS n_fuentes,
           string_agg(DISTINCT source_file, ', ') AS archivos
    FROM {READ}
    WHERE granularidad = 'cliente'
    GROUP BY periodo HAVING count(DISTINCT source_file) > 1
    ORDER BY periodo
""").df()

# el codigo es un correlativo denso 1..N por archivo (no un id estable)
checks["codigo_correlativo"] = con.execute(f"""
    SELECT sum(CASE WHEN mn = 1 AND mx = n AND nd = n THEN 1 ELSE 0 END) AS periodos_correlativos,
           count(*) AS periodos
    FROM (
        SELECT source_file, min(codigo_cliente) AS mn, max(codigo_cliente) AS mx,
               count(*) AS n, count(DISTINCT codigo_cliente) AS nd
        FROM {READ} WHERE granularidad = 'cliente' GROUP BY 1
    )
""").df()

checks["codigo_multi_ubigeo"] = con.execute(f"""
    SELECT count(*) AS codigos FROM (
        SELECT codigo_cliente FROM {READ}
        WHERE granularidad = 'cliente'
        GROUP BY 1 HAVING count(DISTINCT ubigeo) > 1
    )
""").df()

# coherencia ubigeo <-> departamento/provincia/distrito
checks["ubigeo_fuera_puno"] = con.execute(f"""
    SELECT count(*) AS filas, count(DISTINCT ubigeo) AS ubigeos
    FROM {READ}
    WHERE granularidad = 'cliente' AND ubigeo NOT LIKE '21%'
""").df()

checks["ubigeo_multi_provincia"] = con.execute(f"""
    SELECT count(*) AS ubigeos FROM (
        SELECT ubigeo FROM {READ}
        WHERE granularidad = 'cliente'
        GROUP BY 1 HAVING count(DISTINCT provincia) > 1
    )
""").df()

checks["geo_incoherente"] = con.execute(f"""
    WITH dom AS (
        SELECT ubigeo, departamento, provincia, distrito, count(*) AS n
        FROM {READ} WHERE granularidad = 'cliente'
        GROUP BY 1, 2, 3, 4
    ),
    ref AS (
        SELECT ubigeo, departamento, provincia, distrito
        FROM (SELECT *, row_number() OVER (PARTITION BY ubigeo ORDER BY n DESC) AS rn
              FROM dom)
        WHERE rn = 1
    )
    SELECT count(*) AS filas
    FROM {READ} t JOIN ref r USING (ubigeo)
    WHERE t.granularidad = 'cliente'
      AND (t.departamento <> r.departamento
           OR t.provincia <> r.provincia
           OR t.distrito <> r.distrito)
""").df()

# fechas de alta centinela
checks["fecha_alta_centinela"] = con.execute(f"""
    SELECT count(*) FILTER (WHERE fecha_alta < DATE '1950-01-01') AS pre1950,
           count(*) FILTER (WHERE fecha_alta = DATE '1908-09-27') AS en_1908
    FROM {READ} WHERE granularidad = 'cliente'
""").df()

print("Nulos en obligatorios:")
print(checks["nulos_obligatorios"].to_string(index=False))
print("\nNegativos (pre-dedup):", int(checks["negativos"].iloc[0, 0]))
print("Ubigeo invalido:", int(checks["ubigeo_invalido"].iloc[0, 0]))
print("Duplicados (correlativo, periodo):", int(checks["duplicados_cliente_periodo"].iloc[0, 0]))

print("\nPeriodos con mas de un archivo fuente (copias del portal):")
print(checks["fuentes_duplicadas"].to_string(index=False))

print("\n-- Identidad del suministro (pre-dedup) --")
print(f"Periodos con codigo correlativo 1..N: "
      f"{int(checks['codigo_correlativo'].iloc[0, 0])}/{int(checks['codigo_correlativo'].iloc[0, 1])}")
print("Codigos que aparecen con >1 ubigeo:",
      int(checks["codigo_multi_ubigeo"].iloc[0, 0]))

print("\n-- Coherencia geografica (pre-dedup) --")
print("Filas con ubigeo fuera de Puno (21xxxx):",
      int(checks["ubigeo_fuera_puno"].iloc[0, 0]),
      "| ubigeos:", int(checks["ubigeo_fuera_puno"].iloc[0, 1]))
print("Ubigeos con >1 provincia:",
      int(checks["ubigeo_multi_provincia"].iloc[0, 0]))
print("Filas con geo incoherente vs ubigeo dominante:",
      int(checks["geo_incoherente"].iloc[0, 0]))

print("\n-- Fechas de alta (pre-dedup) --")
print("fecha_alta < 1950:", int(checks["fecha_alta_centinela"].iloc[0, 0]),
      "| = 1908-09-27:", int(checks["fecha_alta_centinela"].iloc[0, 1]))

print("\nTarifas:")
print(checks["tarifas"].to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Nulos en obligatorios:
 filas
     0

Negativos (pre-dedup): 18828
Ubigeo invalido: 0
Duplicados (correlativo, periodo): 697626

Periodos con mas de un archivo fuente (copias del portal):
 periodo  n_fuentes                           archivos
  202402          2 ELPU_2024-02.zip, ELPU_2024-06.zip
  202506          2 ELPU_2025-02.zip, ELPU_2025-06.zip

-- Identidad del suministro (pre-dedup) --
Periodos con codigo correlativo 1..N: 36/36
Codigos que aparecen con >1 ubigeo: 261109

-- Coherencia geografica (pre-dedup) --
Filas con ubigeo fuera de Puno (21xxxx): 70 | ubigeos: 2
Ubigeos con >1 provincia: 37
Filas con geo incoherente vs ubigeo dominante: 119000

-- Fechas de alta (pre-dedup) --
fecha_alta < 1950: 23291 | = 1908-09-27: 21906

Tarifas:
tarifa    filas
  BT5B 12424014
  BT5D    72318
   MT4    23809
   BT6    11943
   MT2    11807
   MT3     7799
   BT4     1748
   BT3      699
  BT5A      597
   BT2      132


## 7. Rechazados y deduplicacion

**Hallazgos:**

- El portal publico **archivos duplicados**: `ELPU_2024-06.zip` es copia de
  `ELPU_2024-02.zip` (mismo periodo 202402) y `ELPU_2025-02.zip` es copia de
  `ELPU_2025-06.zip` (mismo periodo 202506). Por eso **no existe data real de
  2024-06 ni 2025-02**.
- Al deduplicar, se conserva el archivo cuyo nombre coincide con el periodo de
  los datos (evita etiquetar 202506 con el archivo "Febrero 2025").
- Hay `consumo_kwh`/`facturacion_soles` **negativos** (creditos, ajustes o
  reconexiones). Se conservan: no son un error de formato.

Filas con errores criticos (sin periodo, sin ubigeo valido o sin consumo) van a
`_rechazados.parquet`. Luego se eliminan duplicados de (correlativo, periodo)
quedando la fila del archivo correcto.

In [10]:
BAD = ("periodo IS NULL OR ubigeo IS NULL "
       "OR NOT regexp_matches(ubigeo, '^[0-9]{6}$') "
       "OR consumo_kwh IS NULL OR facturacion_soles IS NULL")

con.execute(f"""
    COPY (SELECT * FROM {READ} WHERE {BAD})
    TO '{SILVER / '_rechazados.parquet'}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
n_rechazados, = con.execute(
    f"SELECT count(*) FROM read_parquet('{SILVER / '_rechazados.parquet'}')"
).fetchone()

GOOD = f"SELECT * FROM {READ} WHERE NOT ({BAD})"

# desempate: conservar el archivo cuyo nombre coincide con el periodo del dato
con.execute(f"""
    COPY (
        SELECT * EXCLUDE (rn) FROM (
            SELECT *, row_number() OVER (
                PARTITION BY codigo_cliente, periodo
                ORDER BY
                    CASE WHEN source_file LIKE
                        'ELPU_' || substr(periodo::VARCHAR, 1, 4) || '-' ||
                        substr(periodo::VARCHAR, 5, 2) || '%'
                    THEN 0 ELSE 1 END,
                    source_file
            ) AS rn
            FROM ({GOOD})
            WHERE granularidad = 'cliente'
        ) WHERE rn = 1
    ) TO '{SILVER / 'consumo_clientes.parquet'}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.execute(f"""
    COPY (SELECT * FROM ({GOOD}) WHERE granularidad = 'agregado')
    TO '{SILVER / 'consumo_agregado.parquet'}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(f"Rechazados: {n_rechazados:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rechazados: 0


## 8. Resumen mensual continuo (une ambas granularidades)

In [11]:
CLIENTES = str(SILVER / "consumo_clientes.parquet")
AGREGADO = str(SILVER / "consumo_agregado.parquet")

con.execute(f"""
    COPY (
        WITH todo AS (
            SELECT periodo, granularidad, 1::BIGINT AS n,
                   consumo_kwh, facturacion_soles
            FROM read_parquet('{CLIENTES}')
            UNION ALL
            SELECT periodo, granularidad, clientes AS n,
                   consumo_kwh, facturacion_soles
            FROM read_parquet('{AGREGADO}')
        )
        SELECT
            periodo,
            periodo // 100                                          AS anio,
            periodo % 100                                           AS mes,
            granularidad,
            sum(n)                                                  AS n_suministros,
            round(sum(consumo_kwh), 2)                              AS consumo_kwh_total,
            round(sum(facturacion_soles), 2)                        AS facturacion_soles_total,
            round(sum(consumo_kwh) / sum(n), 3)                     AS consumo_kwh_promedio
        FROM todo
        GROUP BY periodo, granularidad
        ORDER BY periodo
    ) TO '{SILVER / 'resumen_mensual.parquet'}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
resumen = con.execute(
    f"SELECT * FROM read_parquet('{SILVER / 'resumen_mensual.parquet'}')"
).df()
print(resumen.head(3).to_string(index=False))
print("...")
print(resumen.tail(3).to_string(index=False))
print(f"\nPeriodos: {resumen['periodo'].min()} -> {resumen['periodo'].max()}  ({len(resumen)} filas)")

 periodo  anio  mes granularidad  n_suministros  consumo_kwh_total  facturacion_soles_total  consumo_kwh_promedio
  202301  2023    1      cliente       336736.0        19738537.44               25946258.8                58.617
  202302  2023    2      cliente       336784.0        18285801.85               23283496.4                54.295
  202303  2023    3      cliente       336988.0        17806276.02               21899917.4                52.839
...
 periodo  anio  mes granularidad  n_suministros  consumo_kwh_total  facturacion_soles_total  consumo_kwh_promedio
  202606  2026    6     agregado       364911.0        25139145.88               34289339.1                68.891
  202607  2026    7     agregado       365881.0        24646740.79               30580034.9                67.363
  202608  2026    8     agregado       366818.0        24772823.63               34968837.5                67.534

Periodos: 202301 -> 202608  (42 filas)


## 9. Reporte final

In [12]:

n_clientes, = con.execute(f"SELECT count(*) FROM read_parquet('{CLIENTES}')").fetchone()
n_agregado, = con.execute(f"SELECT count(*) FROM read_parquet('{AGREGADO}')").fetchone()

FINAL = f"read_parquet(['{CLIENTES}', '{AGREGADO}'], union_by_name=true)"
CL = f"read_parquet('{CLIENTES}')"

final_neg, = con.execute(
    f"SELECT count(*) FROM {FINAL} WHERE consumo_kwh < 0 OR facturacion_soles < 0"
).fetchone()
final_tarifas = con.execute(
    f"SELECT tarifa, count(*) AS filas FROM {FINAL} GROUP BY 1 ORDER BY 2 DESC"
).df()

# validaciones finales sobre la tabla ya deduplicada
fin = {}
fin["corr"] = con.execute(f"""
    SELECT sum(CASE WHEN mn = 1 AND mx = n AND nd = n THEN 1 ELSE 0 END), count(*)
    FROM (SELECT source_file, min(codigo_cliente) AS mn, max(codigo_cliente) AS mx,
                 count(*) AS n, count(DISTINCT codigo_cliente) AS nd
          FROM {CL} GROUP BY 1)
""").fetchone()
fin["multi_ubigeo"] = con.execute(f"""
    SELECT count(*) FROM (
        SELECT codigo_cliente FROM {CL} GROUP BY 1 HAVING count(DISTINCT ubigeo) > 1)
""").fetchone()[0]
fin["fuera_puno"] = con.execute(
    f"SELECT count(*) FROM {CL} WHERE ubigeo NOT LIKE '21%'"
).fetchone()[0]
fin["multi_prov"] = con.execute(f"""
    SELECT count(*) FROM (
        SELECT ubigeo FROM {CL} GROUP BY 1 HAVING count(DISTINCT provincia) > 1)
""").fetchone()[0]
fin["geo_incoh"] = con.execute(f"""
    WITH dom AS (
        SELECT ubigeo, departamento, provincia, distrito, count(*) AS n
        FROM {CL} GROUP BY 1, 2, 3, 4
    ),
    ref AS (
        SELECT ubigeo, departamento, provincia, distrito
        FROM (SELECT *, row_number() OVER (PARTITION BY ubigeo ORDER BY n DESC) AS rn
              FROM dom)
        WHERE rn = 1
    )
    SELECT count(*) FROM {CL} t JOIN ref r USING (ubigeo)
    WHERE t.departamento <> r.departamento
       OR t.provincia <> r.provincia
       OR t.distrito <> r.distrito
""").fetchone()[0]
fin["pre1950"] = con.execute(
    f"SELECT count(*) FROM {CL} WHERE fecha_alta < DATE '1950-01-01'"
).fetchone()[0]
fin["d1908"] = con.execute(
    f"SELECT count(*) FROM {CL} WHERE fecha_alta = DATE '1908-09-27'"
).fetchone()[0]

dup = checks["fuentes_duplicadas"].copy()
if len(dup):
    sel = ",".join(str(int(p)) for p in dup["periodo"])
    kept = con.execute(f"""
        SELECT periodo, any_value(source_file) AS archivo_conservado
        FROM {CL} WHERE periodo IN ({sel}) GROUP BY 1
    """).df()
    dup = dup.merge(kept, on="periodo", how="left")

reporte = {
    "filas_unidas_antes_dedup": int(total),
    "filas_clientes_finales": int(n_clientes),
    "filas_agregado_finales": int(n_agregado),
    "nulos_obligatorios": int(checks["nulos_obligatorios"].iloc[0, 0]),
    "negativos_finales": int(final_neg),
    "ubigeo_invalido": int(checks["ubigeo_invalido"].iloc[0, 0]),
    "duplicados_correlativo_periodo": int(checks["duplicados_cliente_periodo"].iloc[0, 0]),
    "filas_rechazadas": int(n_rechazados),
    "fuentes_duplicadas": dup.to_dict("records"),
    "tarifas_finales": final_tarifas.set_index("tarifa")["filas"].to_dict(),
    "archivos_con_error": procesados.loc[~procesados["columnas_ok"], "archivo"].tolist(),
    "periodos_con_codigo_correlativo": f"{int(fin['corr'][0])}/{int(fin['corr'][1])}",
    "codigos_con_multiples_ubigeos": int(fin["multi_ubigeo"]),
    "filas_ubigeo_fuera_puno": int(fin["fuera_puno"]),
    "ubigeos_con_multiples_provincias": int(fin["multi_prov"]),
    "filas_geo_incoherente": int(fin["geo_incoh"]),
    "fecha_alta_pre1950": int(fin["pre1950"]),
    "fecha_alta_1908_09_27": int(fin["d1908"]),
}
with open(SILVER / "_reporte_silver.json", "w") as f:
    json.dump(reporte, f, indent=2, ensure_ascii=False)

print(json.dumps(reporte, indent=2, ensure_ascii=False))

print("\nArchivos generados:")
for p in sorted(SILVER.glob("*.parquet")):
    print(f"  {p.name:26} {p.stat().st_size/1e6:8.2f} MB")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{
  "filas_unidas_antes_dedup": 12554866,
  "filas_clientes_finales": 11782755,
  "filas_agregado_finales": 74485,
  "nulos_obligatorios": 0,
  "negativos_finales": 18783,
  "ubigeo_invalido": 0,
  "duplicados_correlativo_periodo": 697626,
  "filas_rechazadas": 0,
  "fuentes_duplicadas": [
    {
      "periodo": 202402,
      "n_fuentes": 2,
      "archivos": "ELPU_2024-02.zip, ELPU_2024-06.zip",
      "archivo_conservado": "ELPU_2024-02.zip"
    },
    {
      "periodo": 202506,
      "n_fuentes": 2,
      "archivos": "ELPU_2025-02.zip, ELPU_2025-06.zip",
      "archivo_conservado": "ELPU_2025-06.zip"
    }
  ],
  "tarifas_finales": {
    "BT5B": 11733318,
    "BT5D": 68363,
    "MT4": 22569,
    "BT6": 11348,
    "MT2": 11205,
    "MT3": 7412,
    "BT4": 1665,
    "BT3": 666,
    "BT5A": 569,
    "BT2": 125
  },
  "archivos_con_error": [],
  "periodos_con_codigo_correlativo": "34/34",
  "codigos_con_multiples_ubigeos": 261109,
  "filas_ubigeo_fuera_puno": 66,
  "ubigeos_con_multiples